#### library

In [21]:
from ultralytics import YOLO
import cv2
from tqdm import tqdm

#### reading data

In [2]:
data_yaml = r"D:\Dataset\car_parking_dataset\CarPK.yaml"

#### load model

In [ ]:
model = YOLO("yolov8m.pt")

#### train loop

In [ ]:
model.train(data=data_yaml, epochs=20, batch=4)

#### create demo

In [22]:
final_model = YOLO("runs/detect/train/weights/best.pt")

In [ ]:
input_video = "assets/input_sample.mp4"
output_video = "assets/output_sample.mp4"

CONFIDENCE = 0.5
MAX_FRAMES = None
FPS = 30
CODEC = "mp4v"

cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    print("video not found")
    exit()

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = int(cap.get(cv2.CAP_PROP_FPS)) or FPS

fourcc = cv2.VideoWriter_fourcc(*CODEC)
out = cv2.VideoWriter(output_video, fourcc, FPS, (width, height))

print("processing video...")

frame_count = 0
processed = 0

with tqdm() as pbar:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if MAX_FRAMES and frame_count >= MAX_FRAMES:
            break

        results = final_model.predict(frame, conf=CONFIDENCE, verbose=False)

        annotated_frame = frame.copy()
        num_cars = len(results[0].boxes) if results[0].boxes is not None else 0

        if results[0].boxes is not None:
            for box in results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        text = f"cars: {num_cars}"
        cv2.putText(
            annotated_frame,
            text,
            (30, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            2.0,
            (0, 255, 255),
            5,
            cv2.LINE_AA,
        )

        out.write(annotated_frame)

        frame_count += 1
        processed += 1
        pbar.update(1)

cap.release()
out.release()

print("✅video saved successfully")

#### pt to onnx

In [ ]:
final_model.export(
    format="onnx",
    imgsz=640,
    int8=True,
    data="coco128.yaml",
    dynamic=False
)